In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import annoy

from plotnine import *
import matplotlib.pyplot as plt
import seaborn as sb
from collections import Counter


sc.set_figure_params(figsize=(5,5)) # no blurry figures allowed

pd.set_option('display.max_columns', None)

In [ ]:
%matplotlib inline

# load data

In [ ]:
dr = "/scratch/gpfs/LYDIALYNCH/rc2020"
mice1 = sc.read_10x_mtx(path = dr + '/Mouse1/outs/filtered_feature_bc_matrix', gex_only = False)
mice1

In [ ]:
mice2 = sc.read_10x_mtx(path = dr + '/Mouse2/outs/filtered_feature_bc_matrix', gex_only = False)
mice2

In [ ]:
mice3 = sc.read_10x_mtx(path = dr + '/Mouse3/outs/filtered_feature_bc_matrix', gex_only = False)
mice3

In [ ]:
mice4 = sc.read_10x_mtx(path = dr + '/Mouse4/outs/filtered_feature_bc_matrix', gex_only = False)
mice4

In [ ]:
mice5 = sc.read_10x_mtx(path = dr + '/Mouse5/outs/filtered_feature_bc_matrix', gex_only = False)
mice5

In [ ]:
mice6 = sc.read_10x_mtx(path = dr + '/Mouse6/outs/filtered_feature_bc_matrix', gex_only = False)
mice6

In [ ]:
mice7 = sc.read_10x_mtx(path = dr + '/Mouse7/outs/filtered_feature_bc_matrix', gex_only = False)
mice7

In [ ]:
mice1.obs

In [ ]:
mice1.var

In [ ]:
# save data
out = '/scratch/gpfs/LYDIALYNCH/rc2020/SC_out'
mice1.write(out + '/5.11.25.mice1_exe_qc.h5ad', compression="gzip")

In [ ]:
out = '/scratch/gpfs/LYDIALYNCH/rc2020/SC_out'
mice1 = sc.read_h5ad(out + '/5.11.25.mice1_exe_qc.h5ad')
mice1


In [ ]:
mice1

# qc

In [ ]:
def qc(adata):
    # mitochondrial gene names may start with "Mt-" instead of "mt-""
    adata.var['mt'] = adata.var_names.str.startswith('mt-') # annotate the group of mitochondrial genes as 'mt'
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], multi_panel=True, stripplot=False)
    
    # helpful function to print max/min/mean/median/percentiles for data
    def print_stats(ls0):
        ls = [x for x in ls0 if str(x) != 'nan']
        print("max =", round(max(ls)), "; 0.75 perc =", round(np.percentile(ls, 75)), 
          "; median =", round(np.median(ls)), "; 0.25 perc =", round(np.percentile(ls, 25)),
          "; min =", round(min(ls)),  "; mean =", round(np.mean(ls)))
             # , "; 0.95 perc =", round(np.percentile(ls, 95)), )
    
    print_stats(adata.obs['n_genes_by_counts'])
    print_stats(adata.obs['total_counts'])
    print_stats(adata.obs['pct_counts_mt'])

def qc2(adata, low_n_genes_thres = 1500, high_mt_thres = 12):
    adata.obs["low_n_genes"] = adata.obs["n_genes_by_counts"] <= low_n_genes_thres # abnormal low counts for a cell
    adata.obs["high_mt"] = adata.obs["pct_counts_mt"] >= high_mt_thres
    
    # to pick low genes threshold
    
    ct_pl_col = sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", color="low_n_genes", show=False)
    ct_pl_col.set_xlim(0,max(adata.obs['total_counts']))
    ct_pl_col.set_ylim(0,max(adata.obs['n_genes_by_counts']))
    ct_pl_col
    
    mt_ct_pl = sc.pl.scatter(adata, x="pct_counts_mt", y="n_genes_by_counts", show=False)
    mt_ct_pl.axvline(high_mt_thres, c="r", linestyle='--')
    mt_ct_pl.axhline(low_n_genes_thres, c="r", linestyle='--')

# filtering out mt high and gene count low cells
def qc_filter_cells(adata):
    adata = adata[adata.obs["low_n_genes"] == False]
    print(adata.shape)
    adata = adata[adata.obs["high_mt"] == False]
    print(adata.shape)
    return adata

def qc3(adata, high_ct_thres = 40000):
    adata.obs["high_counts"] = adata.obs["total_counts"] >= high_ct_thres
    
    # to pick low/high counts threshold
    ct_pl_col = sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", color="high_counts", show=False)
    ct_pl_col.set_xlim(0,max(adata.obs['total_counts']))
    ct_pl_col.set_ylim(0,max(adata.obs['n_genes_by_counts']))
    ct_pl_col

    cell_counts = np.sum(adata.X, axis = 1)
    ## are there any unreasonable cell sizes / do we need to filter more? go back up to block 7 with more 
    return ggplot(pd.DataFrame({"cell count": np.array(cell_counts).flatten()}), aes(x="cell count")) + geom_histogram() + geom_vline(xintercept = high_ct_thres, color = "blue")

# filter out high count cells
def qc_filter_cells2(adata):
    print(adata.shape)
    adata = adata[adata.obs["high_counts"] == False]
    print(adata.shape)
    return adata

def remove_doublets(adata):
    # sc.external.pp.scrublet(adata, batch_key="batch")
    print(adata.shape)
    sc.external.pp.scrublet(adata, batch_key=None)
    adata = adata[adata.obs['predicted_doublet'] == False]
    print(adata.shape)
    return adata

In [ ]:
qc(mice1)

In [ ]:
qc2(mice1, low_n_genes_thres = 700, high_mt_thres = 8)

In [ ]:
mice1 = qc_filter_cells(mice1)
mice1

In [ ]:
qc3(mice1, high_ct_thres = 20000)

In [ ]:
mice1= qc_filter_cells2(mice1)
mice1

In [ ]:
mice1

In [ ]:
mice1 = remove_doublets(mice1)
mice1

In [ ]:
mice1

In [ ]:
# save data
out = '/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC'
mice1.write(out + '/19.11.25.mice1_exe_qc_filtercells.h5ad', compression="gzip")

In [ ]:
ls /scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC

# mice 2

In [ ]:
qc(mice2)

In [ ]:
qc2(mice2, low_n_genes_thres = 700, high_mt_thres = 8)

In [ ]:
mice2 = qc_filter_cells(mice2)
mice2

In [ ]:
qc3(mice2, high_ct_thres = 21000)

In [ ]:
mice2= qc_filter_cells2(mice2)
mice2

In [ ]:
mice2 = remove_doublets(mice2)
mice2

In [ ]:
out = '/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC'
mice2.write(out + '/2.12.25.mice2_exe_qc_filtercells.h5ad', compression="gzip")

In [ ]:
ls /scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC

# mice3

In [ ]:
mice3

In [ ]:
qc(mice3)

In [ ]:
plot=sc.pl.violin(mice3, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], multi_panel=True, stripplot=False)
plot.show()    
    

In [ ]:
qc(mice3)

In [ ]:
qc2(mice3, low_n_genes_thres = 700, high_mt_thres = 6)

In [ ]:
mice3 = qc_filter_cells(mice3)
mice3

In [ ]:
qc3(mice3, high_ct_thres = 15000)

In [ ]:
mice3= qc_filter_cells2(mice3)
mice3

In [ ]:
mice3 = remove_doublets(mice3)
mice3

In [ ]:
out = '/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC'
mice3.write(out + '/19.1.26.mice3_exe_qc_filtercells.h5ad', compression="gzip")

In [ ]:
mice4

# mice4

In [ ]:
qc(mice4)

In [ ]:
qc2(mice4, low_n_genes_thres = 700, high_mt_thres = 8)

In [ ]:
mice4= qc_filter_cells(mice4)
mice4

In [ ]:
qc3(mice4, high_ct_thres = 25000)

In [ ]:
mice4 = qc_filter_cells2(mice4)
mice4

In [ ]:
mice4 = remove_doublets(mice4)
mice4

In [ ]:
out = '/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC'
mice4.write(out + '/19.1.26.mice4_exe_qc_filtercells.h5ad', compression="gzip")

# Mice 5 

In [ ]:
mice5

In [ ]:
qc(mice5)

In [ ]:
qc2(mice5, low_n_genes_thres = 700, high_mt_thres = 6)

In [ ]:
mice5= qc_filter_cells(mice5)
mice5

In [ ]:
qc3(mice5, high_ct_thres = 18000)

In [ ]:
mice5= qc_filter_cells2(mice5)
mice5

In [ ]:
mice5 = remove_doublets(mice5)
mice5

In [ ]:
out = '/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC'
mice5.write(out + '/19.1.26.mice5_exetr_qc_filtercells.h5ad', compression="gzip")

# Mice 6

In [ ]:
mice6

In [ ]:
qc(mice6)

In [ ]:
qc2(mice6, low_n_genes_thres = 700, high_mt_thres = 8)

In [ ]:
mice6= qc_filter_cells(mice6)
mice6

In [ ]:
qc3(mice6, high_ct_thres = 22000)

In [ ]:
mice6= qc_filter_cells2(mice6)
mice6

In [ ]:
mice6 = remove_doublets(mice6)
mice6

In [ ]:
out = '/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC'
mice6.write(out + '/19.1.26.mice6_exetr_qc_filtercells.h5ad', compression="gzip")

# Mice 7

In [ ]:
mice7

In [ ]:
qc(mice7)

In [ ]:
qc2(mice7, low_n_genes_thres = 700, high_mt_thres = 8)

In [ ]:
mice7= qc_filter_cells(mice7)
mice7

In [ ]:
qc3(mice7, high_ct_thres = 22000)

In [ ]:
mice7= qc_filter_cells2(mice7)
mice7

In [ ]:
mice7 = remove_doublets(mice7)
mice7

In [ ]:
out = '/scratch/gpfs/LYDIALYNCH/rc2020/SC_out/QC'
mice7.write(out + '/19.1.26.mice7_exetr_qc_filtercells.h5ad', compression="gzip")